In [1]:
#%pip install pandas numpy tqdm sentence-transformers faiss-cpu regex unidecode notebook ipykernel

In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
import faiss

import re
from unidecode import unidecode

In [ ]:
def load_and_prepare_dataset(path):
    """
    Carga el dataset y transforma genres y keywords a listas
    """

    # -------------------------
    # 1. Cargar dataset
    # -------------------------
    df = pd.read_csv(path)

    print(f"Dataset cargado: {df.shape}")

    # -------------------------
    # 2. Función para convertir a lista
    # -------------------------
    def split_pipe_to_list(text):
        if pd.isna(text) or text == "":
            return []
        
        return [item.strip() for item in text.split("|") if item.strip()]

    # -------------------------
    # 3. Aplicar transformación
    # -------------------------
    if "genres" in df.columns:
        df["genres"] = df["genres"].apply(split_pipe_to_list)

    if "keywords" in df.columns:
        df["keywords"] = df["keywords"].apply(split_pipe_to_list)

    print("Transformación completada ✅")

    return df

In [4]:
path = r"C:\Users\juans\OneDrive\Documentos\Maestria en Ingenieria y Analitica de Datos\Proyecto de Grado\CineMate AI\data\raw\tmdb_movies_dataset.csv"

df = load_and_prepare_dataset(path)
df.head()


Dataset cargado: (20000, 9)
Transformación completada ✅


,url,title,release_date,age_rating,runtime_minutes,genres,overview,score,keywords
0,https://www.themoviedb.org/movie/1265609-war-m...,War Machine,03/06/2026,MA 15+,110.0,"[Action, Science Fiction, Thriller]",On one last grueling mission during Army Range...,73,"[military training, military sci-fi, u.s. army..."
1,https://www.themoviedb.org/movie/1290821-shelter,Shelter,01/30/2026,15,107.0,"[Action, Crime, Thriller]",A man living in self-imposed exile on a remote...,67,"[grave, ghosts of the past, apologetic, solita..."
2,https://www.themoviedb.org/movie/799882-the-bluff,The Bluff,02/17/2026,R,102.0,"[Action, Thriller]",When her tranquil life on a remote island is s...,63,"[pirate, sister-in-law, caribbean sea, aggress..."
3,https://www.themoviedb.org/movie/1193501-whistle,Whistle,02/12/2026,R,100.0,"[Horror, Mystery]",A misfit group of unwitting high school studen...,61,"[grave, high school, evil spirit, school, deat..."
4,https://www.themoviedb.org/movie/680493-return...,Return to Silent Hill,01/29/2026,15,106.0,"[Mystery, Drama, Horror]",When James receives a mysterious letter from h...,50,"[painter, monster, letter, wife, supernatural,..."


In [5]:
def generate_field_embeddings(df, model_name="all-MiniLM-L6-v2"):
    """
    Genera embeddings por campo:
    - title
    - overview
    - keywords (join)
    - genres (join)
    
    Retorna un diccionario con arrays numpy
    """

    print("Loading embedding model...")
    model = SentenceTransformer(model_name)

    # -------------------------
    # 1. Preparar textos
    # -------------------------
    titles = df["title"].fillna("").astype(str).tolist()
    overviews = df["overview"].fillna("").astype(str).tolist()

    # join listas → string
    keywords = df["keywords"].apply(lambda x: " ".join(x) if isinstance(x, list) else "").tolist()
    genres = df["genres"].apply(lambda x: " ".join(x) if isinstance(x, list) else "").tolist()

    # -------------------------
    # 2. Generar embeddings
    # -------------------------
    print("\nGenerating title embeddings...")
    emb_title = model.encode(titles, show_progress_bar=True, batch_size=64)

    print("\nGenerating overview embeddings...")
    emb_overview = model.encode(overviews, show_progress_bar=True, batch_size=64)

    print("\nGenerating keywords embeddings...")
    emb_keywords = model.encode(keywords, show_progress_bar=True, batch_size=64)

    print("\nGenerating genres embeddings...")
    emb_genres = model.encode(genres, show_progress_bar=True, batch_size=64)

    # -------------------------
    # 3. Convertir a numpy (por seguridad)
    # -------------------------
    emb_title = np.array(emb_title)
    emb_overview = np.array(emb_overview)
    emb_keywords = np.array(emb_keywords)
    emb_genres = np.array(emb_genres)

    print("\nEmbeddings generados ✅")
    print(f"Shape title: {emb_title.shape}")
    print(f"Shape overview: {emb_overview.shape}")
    print(f"Shape keywords: {emb_keywords.shape}")
    print(f"Shape genres: {emb_genres.shape}")

    return {
        "title": emb_title,
        "overview": emb_overview,
        "keywords": emb_keywords,
        "genres": emb_genres
    }

In [6]:
embeddings = generate_field_embeddings(df)

emb_title = embeddings["title"]
emb_overview = embeddings["overview"]
emb_keywords = embeddings["keywords"]
emb_genres = embeddings["genres"]

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\juans\anaconda3\envs\ai-env\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\juans\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...


Generating keywords embeddings...


Batches:   0%|          | 0/313 [00:00<?, ?it/s]


Generating genres embeddings...


Batches:   0%|          | 0/313 [00:00<?, ?it/s]


Embeddings generados ✅
Shape title: (20000, 384)
Shape overview: (20000, 384)
Shape keywords: (20000, 384)
Shape genres: (20000, 384)


In [7]:
def hybrid_search(
    query,
    df,
    embeddings,
    weights=None,
    top_k=10,
    model_name="all-MiniLM-L6-v2"
):
    """
    Búsqueda con scoring híbrido (multicriterio)
    """

    # -------------------------
    # 1. Pesos por defecto
    # -------------------------
    if weights is None:
        weights = {
            "title": 0.10,
            "genres": 0.05,
            "overview": 0.25,
            "keywords": 0.60
        }

    # -------------------------
    # 2. Modelo
    # -------------------------
    model = SentenceTransformer(model_name)

    # -------------------------
    # 3. Embedding del query
    # -------------------------
    query_emb = model.encode([query])[0]

    # -------------------------
    # 4. Función cosine similarity
    # -------------------------
    def cosine_sim(a, b):
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

    # -------------------------
    # 5. Calcular scores
    # -------------------------
    scores = []

    for i in range(len(df)):

        sim_title = cosine_sim(query_emb, embeddings["title"][i])
        sim_overview = cosine_sim(query_emb, embeddings["overview"][i])
        sim_keywords = cosine_sim(query_emb, embeddings["keywords"][i])
        sim_genres = cosine_sim(query_emb, embeddings["genres"][i])

        # -------------------------
        # Manejo de keywords vacíos
        # -------------------------
        if not df.iloc[i]["keywords"]:  # lista vacía
            adjusted_weights = weights.copy()
            adjusted_weights["keywords"] = 0

            # re-normalizar
            total = sum(adjusted_weights.values())
            for k in adjusted_weights:
                adjusted_weights[k] /= total
        else:
            adjusted_weights = weights

        # -------------------------
        # Score final
        # -------------------------
        score = (
            adjusted_weights["title"] * sim_title +
            adjusted_weights["overview"] * sim_overview +
            adjusted_weights["keywords"] * sim_keywords +
            adjusted_weights["genres"] * sim_genres
        )

        scores.append(score)

    # -------------------------
    # 6. Top K resultados
    # -------------------------
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = df.iloc[top_indices].copy()
    results["score"] = [scores[i] for i in top_indices]

    return results

In [10]:
query = "shounen, anime, supernatural"

results = hybrid_search(query, df, embeddings, top_k=5)

results[["title", "score"]]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,title,score
15079,AYAKA,0.635286
12955,Thus Spoke Kishibe Rohan,0.601484
11076,Shaman King,0.571891
11243,Ushio and Tora,0.568581
16348,Himote House: A Share House of Super Psychic G...,0.567593


In [11]:
def build_faiss_index(embeddings):
    """
    Construye índice FAISS usando embeddings de overview
    """

    emb = embeddings["overview"].astype("float32")

    dimension = emb.shape[1]

    index = faiss.IndexFlatIP(dimension)  # cosine similarity (con normalización)

    # Normalizar vectores (IMPORTANTE para cosine)
    faiss.normalize_L2(emb)

    index.add(emb)

    print(f"FAISS index creado con {index.ntotal} vectores")

    return index

In [12]:
def hybrid_search_faiss(
    query,
    df,
    embeddings,
    faiss_index,
    weights=None,
    top_k=10,
    candidate_k=100,
    model_name="all-MiniLM-L6-v2"
):
    """
    Búsqueda híbrida optimizada con FAISS
    """

    if weights is None:
        weights = {
            "title": 0.10,
            "genres": 0.05,
            "overview": 0.25,
            "keywords": 0.60
        }

    model = SentenceTransformer(model_name)

    # -------------------------
    # 1. Embedding del query
    # -------------------------
    query_emb = model.encode([query]).astype("float32")

    # normalizar para FAISS
    faiss.normalize_L2(query_emb)

    # -------------------------
    # 2. Buscar candidatos (rápido)
    # -------------------------
    distances, indices = faiss_index.search(query_emb, candidate_k)

    candidate_indices = indices[0]

    # -------------------------
    # 3. Función cosine
    # -------------------------
    def cosine_sim(a, b):
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

    # -------------------------
    # 4. Scoring híbrido SOLO en candidatos
    # -------------------------
    scores = []

    for i in candidate_indices:

        sim_title = cosine_sim(query_emb[0], embeddings["title"][i])
        sim_overview = cosine_sim(query_emb[0], embeddings["overview"][i])
        sim_keywords = cosine_sim(query_emb[0], embeddings["keywords"][i])
        sim_genres = cosine_sim(query_emb[0], embeddings["genres"][i])

        # manejo de keywords vacíos
        if not df.iloc[i]["keywords"]:
            adjusted_weights = weights.copy()
            adjusted_weights["keywords"] = 0

            total = sum(adjusted_weights.values())
            for k in adjusted_weights:
                adjusted_weights[k] /= total
        else:
            adjusted_weights = weights

        score = (
            adjusted_weights["title"] * sim_title +
            adjusted_weights["overview"] * sim_overview +
            adjusted_weights["keywords"] * sim_keywords +
            adjusted_weights["genres"] * sim_genres
        )

        scores.append((i, score))

    # -------------------------
    # 5. Ordenar resultados
    # -------------------------
    scores_sorted = sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

    final_indices = [i for i, _ in scores_sorted]
    final_scores = [s for _, s in scores_sorted]

    results = df.iloc[final_indices].copy()
    results["score"] = final_scores

    return results

In [ ]:
faiss_index = build_faiss_index(embeddings)

query = "anime, supernatural, shounen, mecha, fight, based on manga"

results = hybrid_search_faiss(
    query,
    df,
    embeddings,
    faiss_index,
    top_k=5,
    candidate_k=100
)

results[["title", "score"]]

FAISS index creado con 20000 vectores


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,title,score
12955,Thus Spoke Kishibe Rohan,0.579309
16348,Himote House: A Share House of Super Psychic G...,0.570136
17436,Urawa no Usagi-chan,0.570025
10094,Naruto,0.566817
1368,Demon Slayer -Kimetsu no Yaiba- The Movie: Mug...,0.560054
